# Ollama Telugu Website Summarizer

Summarizes any webpage in **Telugu language written with English letters** (Tenglish), using a local Ollama model — no paid API required.

**Models used:** `deepseek-r1:1.5b` via Ollama  
**Prerequisite:** [Install Ollama](https://ollama.com) and run `ollama pull deepseek-r1:1.5b`

In [ ]:
# imports

import os
from dotenv import load_dotenv
from bs4 import BeautifulSoup
import requests
from IPython.display import Markdown, display
from openai import OpenAI

In [ ]:
# Ollama client — points to local server

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "deepseek-r1:1.5b"

In [ ]:
# Website scraper

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_contents(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [ ]:
# Prompts — respond in Telugu using English letters (Tenglish)

system_prompt = """
You are a helpful assistant that summarizes websites in Telugu language using English letters (Tenglish).
Do NOT use Telugu script. Write Telugu words phonetically in English letters only.
Respond in markdown format.
"""

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": f"Summarize this website. If it has news or announcements, include those too.\n\n{website}"}
    ]

In [ ]:
# Summarize function

def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(website)
    )
    display(Markdown(response.choices[0].message.content))

In [ ]:
# Try it — summarize any URL in Tenglish

summarize("https://jcpenney.com")

In [ ]:
summarize("https://bbc.com")